# DATA LOADING PIPELINE (POLARS) - V4

Advanced pipeline with priority-based missing value handling for **LB TOP 5% GUARANTEED**

**Purpose:**
- Load datasets
- Apply priority-based missing value strategies
- cancelled - Create missing flags and filled columns
- Convert data types
- Save processed data for modeling

**Priority Strategy:**
1. **ULTRA Priority:** bz, aw, cc (Δw 10x + test 5%)
2. **HIGH Priority:** az, bl, l, m (Δy=18)
3. **TEST38% Priority:** w, x, y, z (test 38% missing)
4. **CORE Priority:** at, by, ay, cd, ce, cf, al (>5% train+test)
5. **LOW Priority:** Rest 30 features (<1%, safe ffill)

**Expected Result:** 144 new columns (48 flags + 48 filled) = 238 total columns

## 1.1 IMPORTS AND CONFIGURATION

In [1]:
# ============================================
# IMPORTS AND CONFIGURATION
# ============================================
import polars as pl
from pathlib import Path
import time
import numpy as np

# Polars configuration
pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(20)

# Set up paths
base_dir = Path("..")
train_path = base_dir / "data" / "processed" / "train_processed_v1.parquet"  # Float32 without clipping
# Test data options - choose one:
# Option 1: Original test data (Float64)
#test_path = base_dir / "data" / "test.parquet"
# Option 2: Processed test data (Float32) - uncomment to use
test_path = base_dir / "data" / "processed" / "test_processed_v1.parquet"
processed_dir = base_dir / "data" / "processed"
results_dir = base_dir / "Results"
print("✅ Configuration complete")
print(f"📁 Base directory: {base_dir}")

✅ Configuration complete
📁 Base directory: ..


## 1.2 DATA LOADING

In [2]:
# ============================================
# LOAD DATASETS
# ============================================

print("="*60)
print("LOADING DATASETS")
print("="*60)

train_full = pl.read_parquet(train_path)
test_full = pl.read_parquet(test_path)

print(f"✅ Train loaded: {train_full.shape}")
print(f"✅ Test loaded: {test_full.shape}")

# szybki sanity check
print("\n📊 Dtypes (train):")
print(train_full.dtypes[:10])

LOADING DATASETS
✅ Train loaded: (5337414, 94)
✅ Test loaded: (1447107, 92)

📊 Dtypes (train):
[String, Categorical, Categorical, Categorical, Int16, Int16, Float32, Float32, Float32, Float32]


## 1.3 FEATURE GROUPS

In [3]:
# ============================================
# FEATURE GROUPS (PYTHONIC + CORRECT LOGIC)
# ============================================

FEATURE_GROUPS = {
    "ultra": ['feature_bz', 'feature_aw', 'feature_cc'],
    "high": ['feature_az', 'feature_bl', 'feature_l', 'feature_m'],
    "test38": ['feature_w', 'feature_x', 'feature_y', 'feature_z'],
    "core": ['feature_at', 'feature_by', 'feature_ay', 'feature_cd',
             'feature_ce', 'feature_cf', 'feature_al'],
}

# all feature columns
all_features = [c for c in train_full.columns if c.startswith("feature_")]

# detect missing features
missing_cols = [
    c for c in all_features
    if train_full[c].null_count() > 0
]

no_missing_cols = [
    c for c in all_features
    if train_full[c].null_count() == 0
]

# flatten defined groups
high_priority_features = [
    f for group in FEATURE_GROUPS.values() for f in group
]

# 🔥 FIX: LOW only from missing columns
priority_low = [
    f for f in missing_cols if f not in high_priority_features
]

# ============================================
# PRINT SUMMARY
# ============================================

print("🎯 Feature groups summary:")
for k, v in FEATURE_GROUPS.items():
    print(f"   {k.upper():<7} ({len(v)}): {v[:3]}{'...' if len(v)>3 else ''}")

print(f"\n📊 Total features: {len(all_features)}")
print(f"📉 With NaN: {len(missing_cols)}")
print(f"✅ Without NaN: {len(no_missing_cols)}")
print(f"🧊 LOW priority (only NaN features!): {len(priority_low)}")

🎯 Feature groups summary:
   ULTRA   (3): ['feature_bz', 'feature_aw', 'feature_cc']
   HIGH    (4): ['feature_az', 'feature_bl', 'feature_l']...
   TEST38  (4): ['feature_w', 'feature_x', 'feature_y']...
   CORE    (7): ['feature_at', 'feature_by', 'feature_ay']...

📊 Total features: 86
📉 With NaN: 48
✅ Without NaN: 38
🧊 LOW priority (only NaN features!): 30


## 2.1 MISSING FLAGS

## 2.2 FILL TEST38

In [4]:
# FILL TEST38 (NO LEAKAGE - GROUPED FORWARD FILL)
# ============================================

TEST38 = FEATURE_GROUPS["test38"]

# Grouped forward fill by categorical keys
train_full = train_full.with_columns([
    pl.col(c).forward_fill().over(['code', 'sub_code', 'sub_category']).alias(c)
    for c in TEST38
])

test_full = test_full.with_columns([
    pl.col(c).forward_fill().over(['code', 'sub_code', 'sub_category']).alias(c)
    for c in TEST38
    if c in test_full.columns
])

print(f"✅ TEST38 forward-filled (grouped by code/sub_code/sub_category): {len(TEST38)}")

✅ TEST38 forward-filled (grouped by code/sub_code/sub_category): 4


## 2.3 FILL CORE + LOW

In [5]:
# ============================================
# FILL CORE + LOW (NO LEAKAGE - GROUPED FORWARD FILL)
# ============================================

CORE = FEATURE_GROUPS["core"]
LOW = priority_low

fill_cols = CORE + LOW

# Grouped forward fill by categorical keys
train_full = train_full.with_columns([
    pl.col(c).forward_fill().over(['code', 'sub_code', 'sub_category']).alias(c)
    for c in fill_cols
])

test_full = test_full.with_columns([
    pl.col(c).forward_fill().over(['code', 'sub_code', 'sub_category']).alias(c)
    for c in fill_cols
    if c in test_full.columns
])

print(f"✅ CORE + LOW forward-filled (grouped by code/sub_code/sub_category): {len(fill_cols)}")

✅ CORE + LOW forward-filled (grouped by code/sub_code/sub_category): 37


## 2.4 PLACEHOLDER - ULTRA FEATURES

**ULTRA Priority:** Δw 10x + test 5% missing - **Bayesian MCMC ready**

In [ ]:
# ============================================
# PLACEHOLDER - ULTRA FEATURES
# ============================================

# TODO: Manual implementation for ULTRA features
# bz, aw, cc - Δw 10x + test 5% missing

print("🔥 ULTRA features placeholder - manual implementation needed")

## 2.5 PLACEHOLDER - HIGH FEATURES

**HIGH Priority:** Δy=18 impact - **Bayesian MCMC ready**

In [ ]:
# ============================================
# PLACEHOLDER - HIGH FEATURES
# ============================================

# TODO: Manual implementation for HIGH features
# az, bl, l, m - Δy=18 impact

print("🔥 HIGH features placeholder - manual implementation needed")

In [9]:
# ============================================
# MCMC IMPUTATION WITH SAVED MODELS (SAMPLED)
# ============================================
print("="*60)
print("BAYESIAN IMPUTATION - ULTRA & HIGH (SAMPLED DATA)")
print("="*60)

from sklearn.linear_model import BayesianRidge
import numpy as np
import joblib
from pathlib import Path

ULTRA = ['feature_bz', 'feature_aw', 'feature_cc']
HIGH = ['feature_az', 'feature_bl', 'feature_l', 'feature_m']
MCMC_FEATURES = ULTRA + HIGH

model_dir = Path("../models")
model_dir.mkdir(parents=True, exist_ok=True)

# ============================================
# STEP 1: TRAIN MODELS ON SAMPLED TRAIN DATA
# ============================================
print("\n📊 STEP 1: Training Bayesian models on sampled train data...")

models = {}

for feat in MCMC_FEATURES:
    print(f"  Training model for: {feat}")

    # Get rows where feature is not null
    train_clean = train_full.filter(pl.col(feat).is_not_null())

    if len(train_clean) < 100:
        print(f"    ⚠️  Not enough data → forward fill")
        models[feat] = None
        continue

    # 🔥 SAMPLE to reduce memory (use 500k rows max)
    sample_size = min(500_000, len(train_clean))
    train_sample = train_clean.sample(n=sample_size, seed=42)

    print(f"    Using {len(train_sample):,} rows (sampled from {len(train_clean):,})")

    # Select predictors: other features (without target, without weight)
    pred_cols = [c for c in train_full.columns
                 if c.startswith('feature_')
                 and c != feat
                 and c not in MCMC_FEATURES]

    # Add frequency encodings for categories
    for cat in ['code', 'sub_code', 'sub_category']:
        freq_col = f'{cat}_freq'
        freq_map = train_sample.group_by(cat).agg(pl.len().alias(freq_col)).to_dict(as_series=False)
        train_sample = train_sample.with_columns([
            pl.col(cat).replace_strict(freq_map[cat], freq_map[freq_col], default=0).alias(freq_col)
        ])
        pred_cols.append(freq_col)

    # Add temporal features
    train_sample = train_sample.with_columns([
        (pl.col('ts_index') / 3600).alias('ts_norm'),
        (pl.col('horizon') / 25).alias('horizon_norm')
    ])
    pred_cols.extend(['ts_norm', 'horizon_norm'])

    # Prepare data
    X_train = train_sample.select(pred_cols).to_numpy()
    y_train = train_sample.select(feat).to_numpy().ravel()

    # Handle potential NaNs
    if np.any(np.isnan(X_train)):
        X_train = np.nan_to_num(X_train, nan=0.0)

    # Train Bayesian Ridge
    if feat in HIGH:
        # Higher precision for HIGH (Δy=18)
        model = BayesianRidge(max_iter=300, tol=1e-5, alpha_1=1e-5, lambda_1=1e-5)
    else:
        # Standard for ULTRA
        model = BayesianRidge(max_iter=200, tol=1e-4, alpha_1=1e-6, lambda_1=1e-6)

    model.fit(X_train, y_train)

    # Save model and predictors
    models[feat] = {
        'model': model,
        'pred_cols': pred_cols,
        'is_high': feat in HIGH
    }

    # Save to disk
    joblib.dump(model, model_dir / f"{feat}_model.pkl")
    print(f"    ✅ Model saved")

# ============================================
# STEP 2: APPLY TO TEST
# ============================================
print("\n📊 STEP 2: Applying Bayesian imputation to test data...")

for feat in MCMC_FEATURES:
    print(f"  Imputing: {feat}")

    if models[feat] is None:
        test_full = test_full.with_columns([
            pl.col(feat).forward_fill().over(['code', 'sub_code', 'sub_category']).alias(feat)
        ])
        continue

    null_mask = test_full[feat].is_null()

    if null_mask.sum() == 0:
        print(f"    ✅ No nulls in test")
        continue

    # Prepare predictors for test
    test_temp = test_full.clone()

    for cat in ['code', 'sub_code', 'sub_category']:
        freq_col = f'{cat}_freq'
        if freq_col not in test_temp.columns:
            train_freq = train_full.group_by(cat).agg(pl.len().alias(freq_col))
            test_temp = test_temp.join(train_freq, on=cat, how='left')
            test_temp = test_temp.with_columns(pl.col(freq_col).fill_null(0))

    test_temp = test_temp.with_columns([
        (pl.col('ts_index') / 3600).alias('ts_norm'),
        (pl.col('horizon') / 25).alias('horizon_norm')
    ])

    pred_cols = models[feat]['pred_cols']
    X_test = test_temp.filter(null_mask).select(pred_cols).to_numpy()

    if np.any(np.isnan(X_test)):
        X_test = np.nan_to_num(X_test, nan=0.0)

    y_pred = models[feat]['model'].predict(X_test)

    # Create DataFrame with predictions
    null_ids = test_temp.filter(null_mask).select('id').to_numpy().ravel()
    pred_df = pl.DataFrame({
        'id': null_ids,
        f'{feat}_pred': y_pred
    })

    # Join and fill nulls
    test_full = test_full.join(pred_df, on='id', how='left')
    test_full = test_full.with_columns([
        pl.col(feat).fill_null(pl.col(f'{feat}_pred')).alias(feat)
    ])
    test_full = test_full.drop(f'{feat}_pred')

    print(f"    ✅ Imputed {null_mask.sum():,} rows")

# ============================================
# STEP 3: CLEANUP
# ============================================
print("\n📊 STEP 3: Cleaning up temporary columns...")

temp_cols = [c for c in test_full.columns if c.endswith('_freq') or c in ['ts_norm', 'horizon_norm']]
test_full = test_full.drop(temp_cols)

print(f"✅ Bayesian imputation complete for {len(MCMC_FEATURES)} features")
print(f"   ULTRA (3): {ULTRA}")
print(f"   HIGH (4): {HIGH}")


📊 STEP 2: Applying Bayesian imputation to test data...
  Imputing: feature_bz
    ✅ No nulls in test
  Imputing: feature_aw
    ✅ No nulls in test
  Imputing: feature_cc
    ✅ No nulls in test
  Imputing: feature_az
    ✅ No nulls in test
  Imputing: feature_bl
    ✅ No nulls in test
  Imputing: feature_l
    ✅ No nulls in test
  Imputing: feature_m
    ✅ No nulls in test
✅ Bayesian imputation complete


## 3.1 LIGHTGBM MODEL TRAINING

In [10]:
# ============================================
# LIGHTGBM MODEL TRAINING - 4 HORIZON MODELS
# ============================================

print("="*60)
print("TRAINING LIGHTGBM MODELS (PER HORIZON)")
print("="*60)

import lightgbm as lgb
import datetime

# ============================================
# METRIC (bez zmian)
# ============================================

def _clip01(x: float) -> float:
    return float(np.minimum(np.maximum(x, 0.0), 1.0))

def weighted_rmse_score(y_target, y_pred, w) -> float:
    denom = np.sum(w * y_target ** 2)
    ratio = np.sum(w * (y_target - y_pred) ** 2) / denom
    clipped = _clip01(ratio)
    val = 1.0 - clipped
    return float(np.sqrt(val))

# ============================================
# FEATURE COLUMNS (bez zmian - te same dla wszystkich horizonów)
# ============================================

feature_cols = [
    col for col in train_full.columns
    if col.startswith('feature_') or col.endswith('_isnull')
]

print(f"Total features: {len(feature_cols)}")
print(f"Horizons: [1, 3, 10, 25]")

# ============================================
# STORE PREDICTIONS PER HORIZON
# ============================================

horizons = [1, 3, 10, 25]
predictions_by_horizon = {}
models_by_horizon = {}
scores_by_horizon = {}

for horizon in horizons:
    print("\n" + "="*60)
    print(f"HORIZON: {horizon}")
    print("="*60)

    # Filter data by horizon
    train_h = train_full.filter(pl.col('horizon') == horizon)

    # Prepare features
    X_h = train_h.select(feature_cols).to_numpy()
    y_h = train_h.select("y_target").to_numpy().ravel()
    w_h = train_h.select("weight").to_numpy().ravel()

    print(f"Train rows: {len(train_h):,}")
    print(f"Features: {X_h.shape[1]}")

    # Train model
    start_time = time.time()

    model = lgb.LGBMRegressor(
        objective='regression',
        metric='rmse',
        num_leaves=50,
        learning_rate=0.05,
        n_estimators=300,
        max_depth=10,
        min_child_samples=20,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=0.1,
        random_state=42,
        verbose=-1
    )

    model.fit(X_h, y_h, sample_weight=w_h)

    train_time = time.time() - start_time

    # Evaluate
    y_pred_h = model.predict(X_h)
    score_h = weighted_rmse_score(y_h, y_pred_h, w_h)

    print(f"Training time: {train_time:.2f} seconds")
    print(f"Training Score: {score_h:.6f}")

    # Store
    models_by_horizon[horizon] = model
    scores_by_horizon[horizon] = score_h

    # Prepare test predictions for this horizon
    test_h = test_full.filter(pl.col('horizon') == horizon)
    X_test_h = test_h.select(feature_cols).to_numpy()

    preds_h = model.predict(X_test_h)

    predictions_by_horizon[horizon] = {
        'ids': test_h.select('id').to_numpy().ravel(),
        'predictions': preds_h
    }

    print(f"Test rows for horizon {horizon}: {len(test_h):,}")

# ============================================
# SUMMARY
# ============================================

print("\n" + "="*60)
print("SUMMARY - PER HORIZON SCORES")
print("="*60)
for h in horizons:
    print(f"Horizon {h:2d}: {scores_by_horizon[h]:.6f}")

# ============================================
# COMBINE PREDICTIONS FOR SUBMISSION
# ============================================

print("\n" + "="*60)
print("COMBINING PREDICTIONS")
print("="*60)

all_ids = []
all_preds = []

for h in horizons:
    all_ids.extend(predictions_by_horizon[h]['ids'])
    all_preds.extend(predictions_by_horizon[h]['predictions'])

# Convert to numpy arrays and combine in original order
all_ids = np.array(all_ids)
all_preds = np.array(all_preds)

# Create submission dataframe
submission_df = pl.DataFrame({
    'id': all_ids,
    'prediction': all_preds
})

print(f"Submission shape: {submission_df.shape}")
print("Sample predictions:")
print(submission_df.head())

TRAINING LIGHTGBM MODELS (PER HORIZON)
Total features: 86
Horizons: [1, 3, 10, 25]

HORIZON: 1
Train rows: 1,394,653
Features: 86
Training time: 24.35 seconds
Training Score: 0.257953
Test rows for horizon 1: 379,617

HORIZON: 3
Train rows: 1,385,816
Features: 86
Training time: 28.73 seconds
Training Score: 0.267784
Test rows for horizon 3: 376,558

HORIZON: 10
Train rows: 1,337,236
Features: 86
Training time: 36.39 seconds
Training Score: 0.326388
Test rows for horizon 10: 362,057

HORIZON: 25
Train rows: 1,219,709
Features: 86
Training time: 30.91 seconds
Training Score: 0.379675
Test rows for horizon 25: 328,875

SUMMARY - PER HORIZON SCORES
Horizon  1: 0.257953
Horizon  3: 0.267784
Horizon 10: 0.326388
Horizon 25: 0.379675

COMBINING PREDICTIONS
Submission shape: (1447107, 2)
Sample predictions:
shape: (5, 2)
┌─────────────────────────────────┬────────────┐
│ id                              ┆ prediction │
│ ---                             ┆ ---        │
│ str                       

## 3.2 LIGHTGBM MODEL

## 4.1 GENERATE SUBMISSION

## 4.2 SAVE SUBMISSION

In [11]:
# ============================================
# SAVE SUBMISSION FILE
# ============================================
results_dir = base_dir / "Results"
print("="*60)
print("SAVING SUBMISSION FILE")
print("="*60)

# Generate timestamp for filename
import datetime
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
submission_filename = f"lightgbm_benchmark_v4_{timestamp}.csv"
submission_path = results_dir / submission_filename

# Save submission
submission_df.write_csv(submission_path)

print(f"✅ Submission saved: {submission_path}")
print(f"📊 File size: {submission_path.stat().st_size / 1024**2:.2f} MB")
print(f"📝 Records: {len(submission_df)}")

# Verify file exists and show sample
if submission_path.exists():
    print("\n📋 File verification:")
    verify_df = pl.read_csv(submission_path)
    print(f"Loaded shape: {verify_df.shape}")
    print("Sample:")
    print(verify_df.head())
else:
    print("❌ File not found!")

SAVING SUBMISSION FILE
✅ Submission saved: ..\Results\lightgbm_benchmark_v4_20260323_112733.csv
📊 File size: 82.39 MB
📝 Records: 1447107

📋 File verification:
Loaded shape: (1447107, 2)
Sample:
shape: (5, 2)
┌─────────────────────────────────┬────────────┐
│ id                              ┆ prediction │
│ ---                             ┆ ---        │
│ str                             ┆ f64        │
╞═════════════════════════════════╪════════════╡
│ W2MW3G2L__495MGHFJ__PZ9S1Z4V__… ┆ -0.004041  │
│ W2MW3G2L__495MGHFJ__PZ9S1Z4V__… ┆ -0.004346  │
│ W2MW3G2L__495MGHFJ__PZ9S1Z4V__… ┆ -0.004617  │
│ W2MW3G2L__495MGHFJ__PZ9S1Z4V__… ┆ -0.019352  │
│ W2MW3G2L__495MGHFJ__PZ9S1Z4V__… ┆ -0.00536   │
└─────────────────────────────────┴────────────┘


## 5.1 SAVE PROCESSED DATA

In [13]:
# ============================================
# SAVE PROCESSED DATA - V4 (MCMC IMPUTATION)
# ============================================
print("="*60)
print("SAVING PROCESSED DATA - V4 MCMC PIPELINE")
print("="*60)

# Create directory
processed_dir = base_dir / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

# Define paths
train_out_path = processed_dir / "train_processed_v4.parquet"
test_out_path = processed_dir / "test_processed_v4.parquet"

print("💾 Saving enhanced datasets...")

# Save files
train_full.write_parquet(train_out_path)
test_full.write_parquet(test_out_path)

print(f"✅ Train saved: {train_out_path}")
print(f"✅ Test saved: {test_out_path}")

# Get file sizes
train_size = train_out_path.stat().st_size / 1024**2
test_size = test_out_path.stat().st_size / 1024**2

print(f"\n📁 Files created:")
print(f"   Train: {train_size:.1f} MB")
print(f"   Test: {test_size:.1f} MB")

# Summary for v4
feature_cols_count = len([c for c in train_full.columns if c.startswith('feature_')])
print(f"\n🎯 V4 MCMC Pipeline Complete!")
print(f"\n📈 Pipeline Summary:")
print(f"   Total columns: {train_full.shape[1]}")
print(f"   Feature columns: {feature_cols_count}")
print(f"   Imputation methods:")
print(f"      • Bayesian MCMC (ULTRA + HIGH): {len(ULTRA) + len(HIGH)} features")
print(f"      • Forward fill (TEST38, CORE, LOW): {len(TEST38) + len(CORE) + len(LOW)} features")
print(f"\n🚀 v4 ready for LightGBM training!")
print(f"💡 Expected: Better score than v3")

SAVING PROCESSED DATA - V4 MCMC PIPELINE
💾 Saving enhanced datasets...
✅ Train saved: ..\data\processed\train_processed_v4.parquet
✅ Test saved: ..\data\processed\test_processed_v4.parquet

📁 Files created:
   Train: 414.3 MB
   Test: 69.9 MB

🎯 V4 MCMC Pipeline Complete!

📈 Pipeline Summary:
   Total columns: 94
   Feature columns: 86
   Imputation methods:
      • Bayesian MCMC (ULTRA + HIGH): 7 features
      • Forward fill (TEST38, CORE, LOW): 41 features

🚀 v4 ready for LightGBM training!
💡 Expected: Better score than v3
